# XAI-IDS Benchmark - Colab Runner

Clones the `xai-ids-benchmark` repo from GitHub, installs deps, prepares datasets, and runs the full 4 (XAI) x 2 (DL) x 3 (dataset) factorial benchmark from the master's thesis proposal.

Companion files (in repo): `XAI_IDS_Thesis_Proposal.md`, `XAI_IDS_Preregistration_Draft.md`, `XAI_IDS_Gap_Analysis.md`.

Modes:
- `--quick` (default below): 2k rows, 20 epochs, a few minutes on a T4. Verifies wiring.
- Full run: remove `--quick`; hours on T4 / A100.

Datasets require a one-time manual download (UNB/UNSW host them behind a consent form). See the Prepare data cell.

## 1. Configuration

Set your GitHub repo URL and run options.

In [ ]:
GITHUB_REPO = 'https://github.com/SakiburRahman07/xai-ids-benchmark.git'
BRANCH = 'main'
QUICK = True                      # True = smoke run; False = full factorial
DATASETS = None                   # None = all 3; or e.g. ['cicids2017']
MODELS = None                     # None = both; or ['cnn1d','ft_transformer']
METHODS = None                    # None = all enabled; or ['shap','lime']
print(f'Repo: {GITHUB_REPO} (branch {BRANCH}) | Quick: {QUICK}')

## 2. Clone the repo and install dependencies

In [ ]:
import os, subprocess, shutil, sys
REPO_DIR = 'xai-ids-benchmark'
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--depth','1','-b',BRANCH,GITHUB_REPO,REPO_DIR], check=True)
os.chdir(REPO_DIR)
# Auto-detect code location: code may be at repo root OR nested under xai-ids-benchmark/.
if not os.path.exists('scripts/run_benchmark.py') and os.path.exists('xai-ids-benchmark/scripts/run_benchmark.py'):
    os.chdir('xai-ids-benchmark')
CODE_ROOT = os.getcwd()
print('CWD / CODE_ROOT:', CODE_ROOT)
subprocess.run(['ls','-la'], check=False)

In [ ]:
# Install deps. torch is preinstalled on Colab; skip to avoid CUDA mismatch.
# NOTE: rtdl is NOT needed - we use a local FT-Transformer implementation.
import importlib.util
def have(name): return importlib.util.find_spec(name) is not None
pkgs = ['numpy','pandas','scipy','scikit-learn','pyyaml','tqdm','joblib',
        'shap','lime','dice_ml','captum','matplotlib','seaborn','requests','statsmodels']
missing = [p for p in pkgs if not have(p)]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing])
print('deps OK')

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Prepare datasets

**Manual step.** All three datasets sit behind a publisher consent form. Easiest Colab paths:
1. Download zips locally, upload to Google Drive, mount Drive, copy CSVs into `data_raw/<dataset>/`.
2. Or use a Kaggle mirror via the `kaggle` API.

Expected layout:
```
data_raw/cicids2017/*.csv
data_raw/unsw_nb15/UNSW-NB15_*.csv
data_raw/ciciot2023/*.csv
```

In [ ]:
# Option A: mount Google Drive (uncomment).
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p data_raw/cicids2017 data_raw/unsw_nb15 data_raw/ciciot2023
# !cp -r /content/drive/MyDrive/datasets/cicids2017/*.csv data_raw/cicids2017/ 2>/dev/null || true
# !cp -r /content/drive/MyDrive/datasets/unsw_nb15/*.csv  data_raw/unsw_nb15/  2>/dev/null || true
# !cp -r /content/drive/MyDrive/datasets/ciciot2023/*.csv data_raw/ciciot2023/ 2>/dev/null || true

In [ ]:
# Option B: Kaggle API mirror (uncomment + set KAGGLE creds).
# !pip -q install kaggle
# !mkdir -p ~/.kaggle && echo "$KAGGLE_JSON" > ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d chethan111/ids-2017 -p data_raw/cicids2017 --unzip || true

In [ ]:
# Verify datasets are present (prints manual instructions if not).
!python scripts/prepare_data.py --base . || echo 'See manual instructions above.'

## 4. Run the benchmark

Smoke run uses 2k rows / 20 epochs (minutes on T4). Remove `--quick` for the full factorial (hours).

In [ ]:
args = ['scripts/run_benchmark.py','--config-dir','config','--base','.','--out-dir','results']
if QUICK: args.append('--quick')
if DATASETS: args += ['--datasets', *DATASETS]
if MODELS:   args += ['--models', *MODELS]
if METHODS:  args += ['--methods', *METHODS]
print('Running:', ' '.join(args))
subprocess.run([sys.executable, *args], check=False)

## 5. Results & analysis (Friedman/Nemenyi, H2 Wilcoxon)

In [ ]:
import pandas as pd, json, os
tidy = pd.read_csv('results/tidy_metrics.csv')
print('Tidy metric rows:', len(tidy))
tidy.head(20)

In [ ]:
with open('results/friedman_nemenyi.json') as f:
    fn = json.load(f)
for metric, res in fn.items():
    print(f'\n=== {metric} ===')
    print('avg ranks:', res.get('avg_ranks'))
    print('CD:', round(res.get('critical_distance',0),3), '| p:', res.get('p_value'))
    for pair, dec in res.get('pairwise',{}).items():
        print('  ', pair, 'diff=', round(dec['rank_diff'],3), 'sig=', dec['significant'])

In [ ]:
# Critical distance diagram for a chosen metric.
import sys; sys.path.insert(0, os.path.join(CODE_ROOT, 'src'))
from xai_ids_benchmark.analysis.plots import plot_cd
metric_to_plot = 'deletion_auc'  # any scalar metric present in friedman_nemenyi.json
if metric_to_plot in fn:
    plot_cd(fn[metric_to_plot]['avg_ranks'], fn[metric_to_plot]['critical_distance'],
            output_path='results/cd_diagram.png', title=f'CD diagram - {metric_to_plot}')
    from IPython.display import Image, display
    display(Image('results/cd_diagram.png'))

In [ ]:
# H2: stability degradation on UNSW-NB15 (Wilcoxon).
if os.path.exists('results/h2_wilcoxon.json'):
    with open('results/h2_wilcoxon.json') as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print('H2 file not produced (insufficient stability data).')

## 6. Download results

Zip the `results/` folder for download.

In [ ]:
!zip -r results.zip results/ || true
try:
    from google.colab import files; files.download('results.zip')
except Exception as e:
    print('Manual download from the file browser on the left:', e)